# 🛒 تقسيم العملاء وتحليل RFM
**Customer Segmentation & RFM Analysis**

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [1]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "data_analysis/a3_rfm_segmentation"          # مسار المشروع داخل portfolio/
PKGS    = []          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

✓ جاهز — D:\dataanalyst\portfolio\data_analysis\a3_rfm_segmentation


## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| الموضوع | المصدر المقترح |
|---|---|
| RFM Analysis (Recency, Frequency, Monetary) | أي مقال تسويق رقمي + McKinney Ch.10 |
| KMeans Clustering | ISLR Ch.12 / Géron Ch.9 |
| Pandas groupby + aggregation | McKinney Ch.10 |
| Business segmentation strategies | Case studies أونلاين |

## 🎯 بيُستخدم في إيه (استخدام واقعي)
- **فرق التسويق** بتستخدم RFM عشان تقسّم العملاء (VIP / at-risk / lost) وتوجّه الحملات.
- **فرق الـ CRM** بتحدد مين يستاهل خصم ومين محتاج retention campaign.
- **الشركات الكبيرة** بتستخدم RFM + clustering لتخصيص الرسائل والعروض.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

## 1️⃣ تحميل البيانات واستكشافها

In [3]:
df = pd.read_csv("data/ecommerce_transactions.csv", parse_dates=["order_date"])
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDate range:", df["order_date"].min().date(), "->", df["order_date"].max().date())
print("\nUnique customers:", df["customer_id"].nunique())
df.head()

Shape: (15038, 10)

Columns: ['order_id', 'customer_id', 'order_date', 'product_name', 'category', 'unit_price', 'quantity', 'total_amount', 'country', 'payment_method']

Date range: 2022-07-03 -> 2023-12-31

Unique customers: 1600


,order_id,customer_id,order_date,product_name,category,unit_price,quantity,total_amount,country,payment_method
0,ORD113311,CUST2448,2023-10-13,Monitor,Electronics,1166.93,2,2333.86,Canada,PayPal
1,ORD112626,CUST2379,2023-03-23,Perfume,Beauty,75.84,2,151.68,Australia,Credit Card
2,ORD106393,CUST1671,2023-03-30,T-Shirt,Clothing,176.91,1,176.91,USA,PayPal
3,ORD104990,CUST1531,2023-08-21,Monitor,Electronics,422.58,1,422.58,Germany,PayPal
4,ORD112461,CUST2353,2023-01-19,Jacket,Clothing,84.40,3,253.20,Germany,PayPal


## 2️⃣ حساب مقاييس RFM
| المقياس | المعنى | الحساب |
|---|---|---|
| **Recency (R)** | قد إيه العميل اشترى من قريب | عدد الأيام من آخر طلب لتاريخ التحليل |
| **Frequency (F)** | عدد مرات الشراء | عدد الطلبات الفريدة |
| **Monetary (M)** | إجمالي الإنفاق | مجموع total_amount |

In [4]:
analysis_date = df["order_date"].max() + pd.Timedelta(days=1)
print("Analysis date:", analysis_date.date())

rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (analysis_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total_amount", "sum")
).reset_index()

rfm["monetary"] = rfm["monetary"].round(2)
print("\nRFM table shape:", rfm.shape)
print(rfm.describe().round(1))
rfm.head(10)

Analysis date: 2024-01-01

RFM table shape: (1600, 4)
       recency  frequency  monetary
count   1600.0     1600.0    1600.0
mean     186.4        9.4    3676.6
std      113.3        8.8    4013.0
min        1.0        1.0       9.0
25%      101.0        3.0     603.9
50%      168.0        7.0    2381.0
75%      257.2       13.0    5367.8
max      531.0       54.0   23838.6


,customer_id,recency,frequency,monetary
0,CUST1000,379,2,1682.48
1,CUST1001,202,3,267.29
2,CUST1002,111,22,3990.96
3,CUST1003,293,9,763.70
4,CUST1004,336,2,471.19
5,CUST1005,103,16,5522.62
6,CUST1006,187,3,609.97
7,CUST1007,115,7,1781.75
8,CUST1008,118,11,3528.38
9,CUST1009,215,9,3748.53


## 3️⃣ تقييم RFM (RFM Scoring)
بنقسّم كل مقياس لـ 4 مستويات (quartiles):
- **Recency**: الأقل أحسن (اشترى من قريب) → score عالي
- **Frequency**: الأكتر أحسن → score عالي
- **Monetary**: الأكتر أحسن → score عالي

In [5]:
rfm["r_score"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm["rfm_score"] = rfm["r_score"] * 100 + rfm["f_score"] * 10 + rfm["m_score"]
rfm["rfm_total"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

print("RFM score distribution:")
print(rfm[["r_score", "f_score", "m_score", "rfm_total"]].describe().round(1))
rfm.head(10)

RFM score distribution:
       r_score  f_score  m_score  rfm_total
count   1600.0   1600.0   1600.0     1600.0
mean       2.5      2.5      2.5        7.5
std        1.1      1.1      1.1        2.8
min        1.0      1.0      1.0        3.0
25%        1.8      1.8      1.8        5.0
50%        3.0      2.5      2.5        7.0
75%        4.0      3.2      3.2       10.0
max        4.0      4.0      4.0       12.0


,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,rfm_total
0,CUST1000,379,2,1682.48,1,1,2,112,4
1,CUST1001,202,3,267.29,2,1,1,211,4
2,CUST1002,111,22,3990.96,3,4,3,343,10
3,CUST1003,293,9,763.70,1,3,2,132,6
4,CUST1004,336,2,471.19,1,1,1,111,3
5,CUST1005,103,16,5522.62,3,4,4,344,11
6,CUST1006,187,3,609.97,2,1,2,212,5
7,CUST1007,115,7,1781.75,3,2,2,322,7
8,CUST1008,118,11,3528.38,3,3,3,333,9
9,CUST1009,215,9,3748.53,2,3,3,233,8


## 4️⃣ تصنيف العملاء لشرائح (Business Segments)
بنحوّل الأرقام لشرائح بيزنس مفهومة:

In [6]:
def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Big Spenders (Lost)"
    elif r <= 1 and f <= 1:
        return "Lost"
    else:
        return "Need Attention"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

seg_summary = rfm.groupby("segment").agg(
    count=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
).round(1).sort_values("count", ascending=False)

print("Customer Segments:")
print(seg_summary.to_string())

Customer Segments:
                     count  avg_recency  avg_frequency  avg_monetary
segment                                                             
Need Attention         472        211.3            3.1         871.3
Loyal Customers        350        102.2           14.0        5390.7
At Risk                300        245.4           12.9        5029.3
Lost                   160        364.9            1.3         377.3
Champions              150         42.4           26.1       11027.1
New Customers          110         74.2            3.5        1316.8
Big Spenders (Lost)     58        278.2            4.6        3733.1


## 5️⃣ التصوير البياني (Visualizations)

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1 — Segment distribution
seg_counts = rfm["segment"].value_counts()
colors = plt.cm.Set2(range(len(seg_counts)))
axes[0, 0].barh(seg_counts.index, seg_counts.values, color=colors)
axes[0, 0].set_title("عدد العملاء حسب الشريحة", fontsize=13)
axes[0, 0].set_xlabel("عدد العملاء")
for i, v in enumerate(seg_counts.values):
    axes[0, 0].text(v + 5, i, str(v), va="center", fontweight="bold")

# 2 — RFM distributions
for col, ax, color in zip(["recency", "frequency", "monetary"],
                           [axes[0, 1], axes[1, 0], axes[1, 1]],
                           ["#e74c3c", "#2ecc71", "#3498db"]):
    ax.hist(rfm[col], bins=30, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(rfm[col].median(), color="black", linestyle="--", label=f"Median: {rfm[col].median():.0f}")
    ax.set_title(f"توزيع {col.title()}", fontsize=13)
    ax.legend()

plt.tight_layout()
plt.savefig("data/rfm_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: data/rfm_overview.png")

Saved: data/rfm_overview.png


## 6️⃣ تقسيم بـ KMeans (Unsupervised Clustering)
RFM اليدوي حلو للبيزنس، بس نقدر نستخدم KMeans عشان نكتشف مجموعات بشكل تلقائي.

In [8]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["recency", "frequency", "monetary"]])

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, "bo-", linewidth=2)
ax.set_xlabel("عدد المجموعات (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method — اختيار عدد المجموعات الأمثل")
plt.tight_layout()
plt.savefig("data/elbow.png", dpi=120, bbox_inches="tight")
plt.show()

In [9]:
K_BEST = 4
km = KMeans(n_clusters=K_BEST, n_init=10, random_state=42)
rfm["cluster"] = km.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby("cluster").agg(
    count=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean")
).round(1)
print("KMeans Clusters (k=%d):" % K_BEST)
print(cluster_summary.to_string())

KMeans Clusters (k=4):
         count  avg_recency  avg_frequency  avg_monetary
cluster                                                 
0          125         74.0           30.8       13397.7
1          632        141.2            4.6        1470.7
2          396        127.9           15.7        6460.1
3          447        333.3            4.6        1611.2


## 7️⃣ تصوير المجموعات

In [10]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
rfm_2d = pca.fit_transform(rfm_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(rfm_2d[:, 0], rfm_2d[:, 1], c=rfm["cluster"],
                     cmap="Set1", alpha=0.6, s=30, edgecolor="white", linewidth=0.3)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)")
ax.set_title("KMeans Clusters (PCA projection)", fontsize=14)
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("data/clusters_pca.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

PCA explained variance: 96.1%


## 8️⃣ التوصيات التجارية (Business Recommendations)

| الشريحة | الاستراتيجية |
|---|---|
| **Champions** | برنامج ولاء VIP + عروض حصرية + Referral rewards |
| **Loyal Customers** | Upselling + Early access لمنتجات جديدة |
| **New Customers** | Onboarding emails + خصم على الطلب التاني |
| **At Risk** | Win-back campaign + خصم 20% + استبيان رضا |
| **Big Spenders (Lost)** | حملة شخصية مع عرض قوي — الخسارة كبيرة |
| **Lost** | Re-engagement email ثم إزالة من القوائم النشطة |
| **Need Attention** | تحليل أعمق لمعرفة السبب + عروض مخصصة |

In [11]:
print("=== ملخص التحليل ===")
print(f"إجمالي العملاء: {len(rfm):,}")
print(f"\nتوزيع الشرائح:")
for seg, cnt in rfm["segment"].value_counts().items():
    pct = cnt / len(rfm) * 100
    print(f"  {seg:25s} {cnt:5d}  ({pct:.1f}%)")

champions = rfm[rfm["segment"] == "Champions"]
at_risk = rfm[rfm["segment"] == "At Risk"]
print(f"\n💎 Champions: {len(champions)} عميل، متوسط إنفاق {champions['monetary'].mean():,.0f}")
print(f"⚠️  At Risk: {len(at_risk)} عميل — محتاجين حملة استرجاع عاجلة")
print(f"\n📊 عدد المجموعات (KMeans): {K_BEST}")
print("\n✅ التحليل اكتمل بنجاح!")

=== ملخص التحليل ===
إجمالي العملاء: 1,600

توزيع الشرائح:
  Need Attention              472  (29.5%)
  Loyal Customers             350  (21.9%)
  At Risk                     300  (18.8%)
  Lost                        160  (10.0%)
  Champions                   150  (9.4%)
  New Customers               110  (6.9%)
  Big Spenders (Lost)          58  (3.6%)

💎 Champions: 150 عميل، متوسط إنفاق 11,027
⚠️  At Risk: 300 عميل — محتاجين حملة استرجاع عاجلة

📊 عدد المجموعات (KMeans): 4

✅ التحليل اكتمل بنجاح!
